In [0]:
from datetime import datetime
import uuid

run_id = "uc16_" + datetime.utcnow().strftime("%Y%m%d_%H%M%S")
task_name = "build_appointments_clean"

spark.sql(f"""
INSERT INTO poc_health_no_show.ops.pipeline_run_audit
VALUES (
  '{run_id}',
  'uc16_no_show_daily_pipeline',
  '{task_name}',
  'STARTED',
  current_timestamp(),
  NULL,
  NULL,
  NULL
)
""")

In [0]:
%sql
CREATE OR REPLACE TABLE poc_health_no_show.gold.appointment_risk_segments
COMMENT 'Business-ready appointment risk segmentation using transparent, explainable rules. This is an operational prioritization score, not a clinical prediction.'
AS
WITH appointment_features AS (
  SELECT
    a.appointment_id,
    a.patient_pseudo_id,
    a.appointment_date,
    a.appointment_weekday,
    a.age,
    a.gender,
    a.neighbourhood,
    a.sms_received_flag,
    a.no_show_flag,
    a.lead_time_days,
    COALESCE(h.prior_appointments, 0) AS prior_appointments,
    COALESCE(h.prior_no_shows, 0) AS prior_no_shows,
    COALESCE(h.prior_no_show_rate, 0.0) AS prior_no_show_rate
  FROM poc_health_no_show.silver.appointments_clean a
  LEFT JOIN poc_health_no_show.silver.patient_history_features h
    ON a.appointment_id = h.appointment_id
  WHERE a.data_quality_status = 'VALID'
),
scored AS (
  SELECT
    *,
    CASE
      WHEN lead_time_days >= 8 THEN 30
      WHEN lead_time_days BETWEEN 3 AND 7 THEN 20
      ELSE 10
    END
    +
    CASE
      WHEN prior_no_show_rate >= 0.25 THEN 30
      WHEN prior_no_show_rate >= 0.10 THEN 15
      ELSE 0
    END
    +
    CASE
      WHEN NOT sms_received_flag THEN 15
      ELSE 0
    END
    +
    CASE
      WHEN appointment_weekday IN ('Monday', 'Friday') THEN 10
      ELSE 0
    END
    +
    CASE
      WHEN age BETWEEN 18 AND 35 THEN 10
      ELSE 0
    END AS risk_score
  FROM appointment_features
)
SELECT
  appointment_id,
  patient_pseudo_id,
  appointment_date,
  appointment_weekday,
  age,
  gender,
  neighbourhood,
  lead_time_days,
  sms_received_flag,
  prior_appointments,
  prior_no_shows,
  prior_no_show_rate,
  no_show_flag,
  risk_score,
  CASE
    WHEN risk_score >= 70 THEN 'HIGH'
    WHEN risk_score >= 50 THEN 'MEDIUM_HIGH'
    WHEN risk_score >= 30 THEN 'MEDIUM'
    ELSE 'LOW'
  END AS risk_segment,
  concat_ws(
    '; ',
    CASE WHEN lead_time_days >= 8 THEN 'Long lead time' END,
    CASE WHEN lead_time_days BETWEEN 3 AND 7 THEN 'Moderate lead time' END,
    CASE WHEN prior_no_show_rate >= 0.25 THEN 'High prior no-show history' END,
    CASE WHEN prior_no_show_rate >= 0.10 AND prior_no_show_rate < 0.25
      THEN 'Moderate prior no-show history' END,
    CASE WHEN NOT sms_received_flag THEN 'No SMS reminder received' END,
    CASE WHEN appointment_weekday IN ('Monday', 'Friday')
      THEN 'Higher-risk weekday' END,
    CASE WHEN age BETWEEN 18 AND 35 THEN 'Age group associated with higher no-show risk' END
  ) AS risk_reason,
  CASE
    WHEN risk_score >= 70 THEN 'Priority outreach: send reminder and request confirmation'
    WHEN risk_score >= 50 THEN 'Standard outreach: send reminder'
    WHEN risk_score >= 30 THEN 'Monitor: optional reminder'
    ELSE 'Routine scheduling'
  END AS recommended_action,
  'uc16_rules_v1.0' AS risk_model_version,
  current_timestamp() AS _created_at
FROM scored;

In [0]:
%sql
SELECT
  CASE
    WHEN prior_appointments = 0 THEN 'NEW_PATIENT'
    WHEN prior_no_show_rate >= 0.25 THEN 'HIGH_HISTORY_RISK'
    WHEN prior_no_show_rate >= 0.10 THEN 'MEDIUM_HISTORY_RISK'
    ELSE 'LOW_HISTORY_RISK'
  END AS history_profile,
  COUNT(*) AS appointment_count,
  ROUND(AVG(CASE WHEN no_show_flag THEN 1.0 ELSE 0.0 END) * 100, 2) AS no_show_rate_pct
FROM poc_health_no_show.silver.appointments_clean a
JOIN poc_health_no_show.silver.patient_history_features h
  ON a.appointment_id = h.appointment_id
GROUP BY history_profile
ORDER BY appointment_count DESC;

In [0]:
%sql
SELECT
  risk_segment,
  COUNT(*) AS appointment_count,
  ROUND(AVG(CASE WHEN no_show_flag THEN 1.0 ELSE 0.0 END) * 100, 2) AS actual_no_show_rate_pct
FROM poc_health_no_show.gold.appointment_risk_segments
GROUP BY risk_segment
ORDER BY actual_no_show_rate_pct DESC;

In [0]:
%sql
UPDATE poc_health_no_show.gold.appointment_risk_segments
SET
  age = 0,
  risk_reason = concat_ws(
    '; ',
    risk_reason,
    'Age override applied: source age was invalid and standardized to 0 for operational analysis'
  ),
  risk_model_version = 'uc16_rules_v1.1_with_age_override',
  _created_at = current_timestamp()
WHERE appointment_id IN (
  SELECT appointment_id
  FROM poc_health_no_show.silver.appointments_clean
  WHERE data_quality_status = 'INVALID_AGE'
);

In [0]:
spark.sql(f"""
INSERT INTO poc_health_no_show.ops.pipeline_run_audit
VALUES (
  '{run_id}',
  'uc16_no_show_daily_pipeline',
  '{task_name}',
  'SUCCESS',
  current_timestamp(),
  current_timestamp(),
  NULL,
  NULL
)
""")